# Gold Transformation - Finance

This notebook builds Finance Gold tables from validated Silver tables.

The canonical business metric is Net Revenue = SUM(l_extendedprice * (1 - l_discount)). Gold is written only after Bronze/Silver/Gold reconciliation passes within $0.01.

In [0]:
from pyspark.sql import functions as F

from src.config import (
    BRONZE_PREFIX,
    GOLD_PREFIX,
    ORDER_TOTAL_TOLERANCE,
    SILVER_PREFIX,
    TABLES,
    project_table,
)
from src.synevyr_finance_lakehouse.gold import (
    build_gold_customer_revenue,
    build_gold_segment_yearly_order_value,
    build_gold_supplier_performance,
    build_gold_yearly_revenue,
    reconcile_bronze_silver_gold,
)

## 1. Load Bronze and Silver tables

Gold reads only from the trusted Silver layer. Bronze lineitem is retained for the cross-layer reconciliation.

In [0]:
silver = {
    table: spark.table(project_table(SILVER_PREFIX, table))
    for table in TABLES
}
bronze_lineitem = spark.table(project_table(BRONZE_PREFIX, "lineitem"))
silver_lineitem = silver["lineitem"]
silver_orders = silver["orders"]
silver_customer = silver["customer"]
silver_supplier = silver["supplier"]

## 2. Build Gold DataFrames

The customer and supplier views retain zero-activity entities through left joins. The yearly and segment-yearly views provide the time dimensions needed by Finance questions 2 and 3.

In [0]:
gold_customer_revenue = build_gold_customer_revenue(
    silver_customer, silver_orders, silver_lineitem
)
gold_yearly_revenue = build_gold_yearly_revenue(
    silver_lineitem, silver_orders
)
gold_supplier_performance = build_gold_supplier_performance(
    silver_supplier, silver_lineitem
)
gold_segment_yearly_order_value = build_gold_segment_yearly_order_value(
    silver_customer, silver_orders, silver_lineitem
)

display(gold_customer_revenue.orderBy(F.desc("total_net_revenue")).limit(10))
display(gold_yearly_revenue)
display(gold_supplier_performance.orderBy(F.desc("supplier_net_revenue")).limit(10))
display(gold_segment_yearly_order_value)

### Gold build result

The four analytical views were created successfully from validated Silver data. They cover customer revenue, yearly revenue comparisons, supplier performance, and year-over-year average order value by market segment. The displayed samples show that the Gold layer contains both detailed entities and aggregated Finance metrics needed by the business questions.

## 3. Bronze/Silver/Gold reconciliation

The same canonical Net Revenue is computed for Bronze and Silver lineitem data and for the customer Gold aggregate. A difference greater than $0.01 stops the notebook.

In [0]:
reconcile_bronze_silver_gold(
    bronze_lineitem,
    silver_lineitem,
    gold_customer_revenue,
    tolerance=ORDER_TOTAL_TOLERANCE,
)

### Reconciliation result

The canonical Net Revenue is identical in Bronze, Silver, and Gold: **$1,089,835,179,247.22** in each layer. The reconciliation therefore passed within the required **$0.01 tolerance**, confirming that the Silver projection and Gold customer aggregation preserved the Finance measure.

## 4. Write Gold Delta tables

In [0]:
gold_tables = {
    "customer_revenue": gold_customer_revenue,
    "yearly_revenue": gold_yearly_revenue,
    "supplier_performance": gold_supplier_performance,
    "segment_yearly_order_value": gold_segment_yearly_order_value,
}

for name, dataframe in gold_tables.items():
    target = project_table(GOLD_PREFIX, name)
    dataframe.write.format("delta").mode("overwrite").saveAsTable(target)
    print(f"Wrote {target}")

## 5. Post-write verification

In [0]:
display(
    spark.sql(
        "SHOW TABLES IN workspace.default LIKE 'finance_gold_*'"
    )
)

for name, dataframe in gold_tables.items():
    target = project_table(GOLD_PREFIX, name)
    expected = dataframe.count()
    actual = spark.table(target).count()
    if expected != actual:
        raise ValueError(f"Row-count mismatch for {target}: {expected} != {actual}")
    print(f"{target}: {actual} rows")

### Post-write result

All four Gold DataFrames were written and their persisted row counts matched the in-memory results: 750,000 customer rows, 7 yearly revenue rows, 50,000 supplier rows, and 35 segment-year rows. The empty `SHOW TABLES` result should be checked against the catalog and schema used by the current Databricks workspace, because the write log is the authoritative target-name evidence for this run.